# Case-Study Activity: The I-94 "Traffic Pulse"
**UE24CS342AA9 · Unit 3 · PPT 5: Animation, Motion and Temporal Visualization · Duration: ~28 minutes**

**Dataset:** Metro Interstate Traffic Volume (Kaggle) — https://www.kaggle.com/datasets/anshtanwar/metro-interstate-traffic-volume
Hourly westbound traffic on Interstate 94 between Minneapolis and St Paul (2012–2018), with weather and holidays.

---
### 📋 The case
You have just joined the **Minnesota Department of Transportation (MnDOT)** as a data-visualization analyst. Your manager needs two things:

1. **For the road-works engineers:** evidence for *when* a daytime lane closure on I-94 would disrupt the fewest drivers.
2. **For the public lobby screen:** an animated *"Traffic Pulse"* that shows how a typical day's traffic changes through the year.

A vendor has also offered to cut storage costs by **recording traffic only once every 25 hours**. You need to advise on that too.

Each part has **💻 coding tasks** (checked automatically) and **✍️ reflection questions** (write 2–3 sentences each).



## Part 0 — Setup (1 min): run both cells, don't edit

In [ ]:
%pip install -q kagglehub plotly

In [ ]:
import os, glob, calendar, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, plotly.express as px
warnings.filterwarnings("ignore")

if os.path.exists("Metro_Interstate_Traffic_Volume.csv"):
    path = "Metro_Interstate_Traffic_Volume.csv"
else:
    import kagglehub
    path = sorted(glob.glob(os.path.join(kagglehub.dataset_download("anshtanwar/metro-interstate-traffic-volume"), "**", "*.csv"), recursive=True))[0]

raw = pd.read_csv(path); raw.columns = raw.columns.str.strip().str.lower()
for kw in [dict(format="%Y-%m-%d %H:%M:%S"), dict(format="%d-%m-%Y %H:%M"), dict(format="%d-%m-%Y %H:%M:%S"), dict(format="mixed", dayfirst=True)]:
    t = pd.to_datetime(raw["date_time"], errors="coerce", **kw)
    if t.notna().mean() > 0.99: raw["date_time"] = t; break
raw["holiday"] = raw["holiday"].fillna("None").astype(str).str.strip().replace({"": "None", "nan": "None"})
print(f"raw: {len(raw):,} rows, {raw.date_time.min()} → {raw.date_time.max()}")
raw.head()

In [ ]:
# ===================== AUTO-CHECKER — run, do not edit =====================
_S = {}
def _r(k, ok, good, hint):
    _S[k] = bool(ok); print(("✅ " + good) if ok else ("❌ Not yet — hint: " + hint))
def _exp_clean():
    t = raw.drop_duplicates("date_time").sort_values("date_time").set_index("date_time")
    h = t.loc[t.holiday != "None", "holiday"]; hd = pd.Series(h.values, index=h.index.normalize()); hd = hd[~hd.index.duplicated()]
    t["holiday_day"] = t.index.normalize().map(hd).fillna("None"); return t
_T = _exp_clean()

def check_A(n_dups, tr, modal_interval_h, longest_gap_days):
    g = _T.index.to_series().diff().dropna()
    if n_dups != raw.date_time.duplicated().sum(): return _r("A", 0, "", "count duplicated timestamps in raw['date_time'] with .duplicated().sum().")
    if not (isinstance(tr, pd.DataFrame) and len(tr) == len(_T) and tr.index.is_monotonic_increasing):
        return _r("A", 0, "", "drop duplicate timestamps, sort, and set 'date_time' as the index.")
    if abs(modal_interval_h - g.mode()[0].total_seconds() / 3600) > 1e-9: return _r("A", 0, "", "take the MODE of the differences between consecutive timestamps, in hours.")
    if abs(longest_gap_days - g.max().total_seconds() / 86400) > 0.01: return _r("A", 0, "", "longest gap = max of the differences, converted to days.")
    _r("A", 1, f"Part A correct — modal interval {modal_interval_h:g} h, longest gap {longest_gap_days:.0f} days.", "")

def check_B(nyquist_max_interval, aliased_period, apparent_period_25h):
    try:
        ok1 = abs(nyquist_max_interval(24) - 12) < 1e-9 and abs(nyquist_max_interval(15) - 7.5) < 1e-9
        ok2 = abs(aliased_period(50, 60) - 300) < 1e-6 and abs(aliased_period(24, 23) - 552) < 1e-6 and abs(aliased_period(24, 25) - 600) < 1e-6
    except Exception as e: return _r("B", 0, "", f"error: {e}")
    if not ok1: return _r("B", 0, "", "Nyquist: fs > 2f, so the sampling interval must be < period / 2.")
    if not ok2: return _r("B", 0, "", "aliased frequency = |f − fs| (f = 1/period, fs = 1/interval); period = 1 / that.")
    if abs(apparent_period_25h - 600) > 1e-6: return _r("B", 0, "", "call your aliased_period with the 24-h cycle and a 25-h interval.")
    _r("B", 1, "Part B correct — a 24-h cycle sampled every 25 h looks like a 600-h (25-day) cycle!", "")

def check_C(daily, ma7_centred, ma7_trailing, points_per_pixel):
    d = _T["traffic_volume"].loc["2017"].resample("D").sum(min_count=1)
    if not (isinstance(daily, pd.Series) and len(daily) == len(d) and np.allclose(daily.fillna(-1), d.fillna(-1))):
        return _r("C", 0, "", "daily = tr['traffic_volume'].loc['2017'].resample('D').sum(min_count=1).")
    if not np.allclose(ma7_centred.fillna(-1), d.rolling(7, center=True).mean().fillna(-1)): return _r("C", 0, "", "centred MA: .rolling(7, center=True).mean().")
    if not np.allclose(ma7_trailing.fillna(-1), d.rolling(7).mean().fillna(-1)): return _r("C", 0, "", "trailing MA: .rolling(7).mean() (no center).")
    if abs(points_per_pixel - len(_T.loc["2017"]) / 1000) > 1e-6: return _r("C", 0, "", "number of hourly rows in 2017 divided by 1000 px.")
    _r("C", 1, f"Part C correct — hourly 2017 on 1000 px = {points_per_pixel:.1f} points per pixel.", "")

def check_D(weekday_profile, holiday_profile, best_closure_hour):
    w = _T[(_T.index.dayofweek < 5) & (_T.holiday_day == "None")].groupby(_T.index.hour[(_T.index.dayofweek < 5) & (_T.holiday_day == "None")])["traffic_volume"].mean()
    h = _T[_T.holiday_day != "None"]; h = h.groupby(h.index.hour)["traffic_volume"].mean()
    try:
        if not np.allclose(np.asarray(weekday_profile), w.values): return _r("D", 0, "", "weekday = Mon–Fri (dayofweek < 5) AND holiday_day == 'None'; then group by hour and take the mean.")
        if not np.allclose(np.asarray(holiday_profile), h.values): return _r("D", 0, "", "holiday rows: holiday_day != 'None'; group by hour; mean.")
    except Exception as e: return _r("D", 0, "", f"error: {e}")
    if best_closure_hour != int(w.loc[9:15].idxmin()): return _r("D", 0, "", "restrict weekday_profile to hours 9–15 (.loc[9:15]) and take .idxmin().")
    _r("D", 1, f"Part D correct — the quietest daytime start hour on normal weekdays is {best_closure_hour}:00.", "")

def check_E(frame_budget_ms, smoothstep, n_frames, duration_s, compression, fig):
    try:
        ok1 = abs(frame_budget_ms(60) - 1000/60) < 1e-9 and abs(frame_budget_ms(24) - 1000/24) < 1e-9
        ok2 = abs(smoothstep(0.25) - 0.15625) < 1e-9 and smoothstep(0) == 0 and abs(smoothstep(1) - 1) < 1e-9
    except Exception as e: return _r("E", 0, "", f"error: {e}")
    if not ok1: return _r("E", 0, "", "frame budget = 1000 / fps milliseconds.")
    if not ok2: return _r("E", 0, "", "smoothstep s(u) = 3u² − 2u³.")
    if (n_frames, round(duration_s, 6), round(compression)) != (540, 18.0, 4800): return _r("E", 0, "", "24 hours × 0.75 s = duration; × 30 fps = frames; compression = 3600 s of data / 0.75 s.")
    try:
        yr = fig.layout.yaxis.range; nfr = len(fig.frames)
    except Exception: return _r("E", 0, "", "fig must be a plotly express figure with animation_frame.")
    if nfr != 12: return _r("E", 0, "", "use animation_frame='Month' so there is one frame per month (12).")
    if yr is None: return _r("E", 0, "", "fix the y-axis with range_y=[0, 7500] so bars stay comparable across frames (congruence!).")
    _r("E", 1, "Part E correct — 540 frames, 18 s, 4800:1, fixed axis, 12 frames.", "")

def summary():
    for k in "ABCDE": print(f"Part {k}: {'✅' if _S.get(k) else '❌'}")
    print(f"Coding score: {sum(_S.values())}/5   (+ reflection questions, graded by instructor)")
print("Auto-checker ready.")

---
## Part A — Understand the temporal data 
Before you visualize anything, check the **temporal structure** of the data.

**A1.** Compute:
- `n_dups`: the number of **duplicate timestamps** in `raw` (several weather labels can share an hour)
- `tr`: a clean table with **one row per hour**, sorted, with `date_time` as the **index**
- `modal_interval_h`: the **most common** gap between consecutive rows, in hours
- `longest_gap_days`: the **longest** gap in the record, in days

In [ ]:
n_dups = int(raw["date_time"].duplicated().sum())
tr = (raw.drop_duplicates(subset="date_time")
        .sort_values("date_time")
        .set_index("date_time"))

gaps = tr.index.to_series().diff().dropna()
modal_interval_h = gaps.mode().iloc[0].total_seconds() / 3600
longest_gap_days = gaps.max().total_seconds() / 86400

check_A(n_dups, tr, modal_interval_h, longest_gap_days)

In [ ]:
# (given) Holiday names appear only on the midnight row, so spread them to the whole day. Then plot the record.
_h = tr.loc[tr.holiday != "None", "holiday"]; _hd = pd.Series(_h.values, index=_h.index.normalize()); _hd = _hd[~_hd.index.duplicated()]
tr["holiday_day"] = tr.index.normalize().map(_hd).fillna("None")
tr["traffic_volume"].resample("D").sum(min_count=1).plot(figsize=(14, 3), lw=0.6, title="Daily vehicles — note the gap"); plt.show()

**A2.** Time is a **KEY attribute** here because each timestamp identifies and orders an hourly traffic observation. Time would be a VALUE attribute if the measured quantity itself were a time, such as the number of minutes required to clear a traffic incident.

**A3.** The observations form a **discrete-time series** because traffic is recorded at hourly timestamps. Traffic volume is a count, so it is discrete-valued; a line is suitable for showing its temporal trend. Use bars for the number of holiday days per month because these are monthly category counts.

**A4.** An animation that ignores the gap could imply that traffic was continuously observed and that the change between years was gradual. Show the missing interval explicitly as a blank/gap, label the data outage, and avoid interpolating across the long gap.

---
## Part B — The vendor's sampling proposal (5 min) 
The vendor wants to record **one sample every 25 hours**. Traffic has a strong **24-hour cycle**.

**B1.** Implement:
- `nyquist_max_interval(period_h)`: the **largest** sampling interval that avoids aliasing (Nyquist: $f_s > 2f$)
- `aliased_period(true_period, sample_interval)`: the **apparent period**, using aliased frequency $f_a = |f - f_s|$ for $f_s/2 < f < 3f_s/2$ (the case in this activity)

Then compute `apparent_period_25h` for the vendor's plan.

In [ ]:
def nyquist_max_interval(period_h):
    # Nyquist requires sampling frequency strictly greater than 2f.
    # The limiting interval is period / 2; use a slightly shorter interval in practice.
    return period_h / 2

def aliased_period(true_period, sample_interval):
    f = 1 / true_period
    fs = 1 / sample_interval
    fa = abs(f - fs)
    return 1 / fa

apparent_period_25h = aliased_period(24, 25)

check_B(nyquist_max_interval, aliased_period, apparent_period_25h)

In [ ]:
# (given) What the vendor's data would look like for spring 2017
seg = tr["traffic_volume"].loc["2017-03-01":"2017-05-31"].asfreq("h").interpolate(limit=3)
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(seg.index, seg.values, c="lightgrey", lw=0.4, label="true hourly")
ax.plot(seg.index[::25], seg.values[::25], "-o", c="red", ms=3, label="vendor: every 25 h")
ax.set_title(f"Sampling every 25 h turns the 24-h cycle into a fake ~{apparent_period_25h/24:.0f}-day wave"); ax.legend(); plt.show()

**B2.** The traffic has a true 24-hour cycle, but sampling every 25 hours is too slow to satisfy the Nyquist requirement for that cycle. The sampling rate is below twice the daily-cycle frequency, so aliasing makes the daily pattern appear as a much slower cycle of about 600 hours (25 days). The apparent 25-day rise and fall is an artefact of sampling, not a real traffic cycle.

**B3.** The theoretical Nyquist limit is an interval shorter than 12 hours for a 24-hour cycle. Sampling exactly every 12 hours is the boundary case and is not safely above the Nyquist rate; recommend an interval clearly shorter than 12 hours, preferably retaining hourly observations for this dataset.

---
## Part C — Aggregation and smoothing (5 min) *(slides 26–28)*
The engineers want a clean view of **2017**.

💻 **C1.** Compute:
- `daily`: **daily total** vehicles in 2017 (use `min_count=1`, so days with no data become NaN, not 0)
- `ma7_centred`: a **7-day centred** moving average of `daily`
- `ma7_trailing`: a **7-day trailing** moving average (the default `.rolling`)
- `points_per_pixel`: if every **hourly** row of 2017 were plotted on a **1000-pixel**-wide chart

In [ ]:
daily = tr["traffic_volume"].loc["2017"].resample("D").sum(min_count=1)
ma7_centred = daily.rolling(7, center=True).mean()
ma7_trailing = daily.rolling(7).mean()
points_per_pixel = len(tr.loc["2017"]) / 1000

check_C(daily, ma7_centred, ma7_trailing, points_per_pixel)

In [ ]:
# (given)
fig, ax = plt.subplots(1, 2, figsize=(16, 4))
ax[0].plot(daily.index, daily / 1e3, c="lightgrey", label="daily"); ax[0].plot(ma7_centred.index, ma7_centred / 1e3, lw=2, label="7-day centred")
ax[0].plot(daily.index, daily.rolling(30, center=True).mean() / 1e3, lw=2, c="red", label="30-day centred"); ax[0].legend(); ax[0].set_title("2017 daily traffic (thousands)")
z = slice("2017-06-20", "2017-07-20")
ax[1].plot(daily.loc[z] / 1e3, "o-", c="lightgrey", ms=3); ax[1].plot(ma7_centred.loc[z] / 1e3, lw=2, label="centred"); ax[1].plot(ma7_trailing.loc[z] / 1e3, lw=2, ls="--", label="trailing")
ax[1].axvline(pd.Timestamp("2017-07-04"), c="k", ls=":"); ax[1].legend(); ax[1].set_title("Zoom: around 4 July"); ax[1].tick_params(axis="x", rotation=45); plt.tight_layout(); plt.show()

**C2.** A 7-day trailing average is centred approximately 3 days before its label, so it lags by about 3 days relative to a centred 7-day average. This matters because a reader may attribute a dip or recovery to a later date than when it actually occurred.

**C3.** Use raw daily values to compare individual weekdays, ideally across multiple weeks; a 7-day average is useful for reducing day-to-day noise but hides the weekday pattern; a 30-day average emphasizes the broader trend and hides short events and weekly variation. `points_per_pixel` indicates how many hourly observations compete for each horizontal pixel; when it is greater than 1, individual points cannot all be visually distinguished, so aggregation or smoothing is appropriate for an overview.

---
## Part D — Filtering and events: choose the closure hour (4 min) *(slides 24, 29)*
Crews can only work in **daytime**, starting between **9:00 and 15:00**. Find the **quietest start hour on a normal weekday**, and compare it with holidays.

💻 **D1.** Compute:
- `weekday_profile`: mean traffic **by hour (0–23)** for **Mon–Fri non-holiday** rows
- `holiday_profile`: mean traffic by hour for rows where `holiday_day != "None"`
- `best_closure_hour`: the hour between **9 and 15** (inclusive) where `weekday_profile` is lowest

In [ ]:
wk = tr[(tr.index.dayofweek < 5) & (tr["holiday_day"] == "None")]
weekday_profile = wk.groupby(wk.index.hour)["traffic_volume"].mean()

hol = tr[tr["holiday_day"] != "None"]
holiday_profile = hol.groupby(hol.index.hour)["traffic_volume"].mean()

best_closure_hour = int(weekday_profile.loc[9:15].idxmin())

check_D(weekday_profile, holiday_profile, best_closure_hour)

In [ ]:
# (given)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(weekday_profile.index, weekday_profile.values, "-o", ms=3, label="normal weekdays")
ax.plot(holiday_profile.index, holiday_profile.values, "-o", ms=3, label="holidays")
ax.axvspan(9, 15, color="green", alpha=0.08, label="allowed start window"); ax.axvline(best_closure_hour, c="green", ls="--")
ax.set(xlabel="hour", ylabel="mean vehicles/hour", title=f"Filtered profiles — quietest weekday start: {best_closure_hour}:00"); ax.legend(); plt.show()

**D2.** Filtering retains only normal Monday–Friday observations and removes weekends and holidays; aggregation then replaces the individual observations with one mean value for each hour. This makes the typical hourly profile easier to compare, but hides variation and unusual high-traffic days. A risk is that the mean can conceal high-percentile congestion or incident-related spikes, so the closure decision should also consider distributions/percentiles and day-to-day variability.

---
## Part E — Design the animated *Traffic Pulse* for the lobby screen (6 min) 
**Design brief:** the screen animates **24 hourly states** of a typical weekday. Each hour is shown for **0.75 s** at **30 fps**, with a smooth transition between hours.

💻 **E1.** Implement and compute:
- `frame_budget_ms(fps)`: milliseconds available per frame
- `smoothstep(u)`: the ease-in-out function $3u^2-2u^3$
- `n_frames`, `duration_s`: for the brief above
- `compression`: seconds of **data** per second of **animation** (one hour of data per 0.75 s)

In [ ]:
def frame_budget_ms(fps):
    return 1000 / fps

def smoothstep(u):
    return 3 * u**2 - 2 * u**3

duration_s = 24 * 0.75
n_frames = int(duration_s * 30)
compression = 3600 / 0.75

print(f"Frame budget at 30 fps: {frame_budget_ms(30):.2f} ms")
print(f"smoothstep(0.25): {smoothstep(0.25):.5f}")
print(f"Animation duration: {duration_s:.1f} s")
print(f"Total frames: {n_frames}")
print(f"Data-time / animation-time compression: {compression:.0f}:1")

💻 **E2.** A second version shows how the **weekday profile changes month by month** in 2017. Build `fig` with `px.bar` using **`animation_frame="Month"`**, and **fix the y-axis** so bars stay comparable across frames (congruence).

In [ ]:
# (given) monthly weekday profiles for 2017
wk17 = wk.loc["2017"]
mp = wk17.groupby([wk17.index.month.rename("month"), wk17.index.hour.rename("hour")])["traffic_volume"].mean().rename("vehicles").reset_index()
mp["Month"] = mp["month"].map(lambda m: calendar.month_abbr[m])
MONTHS = [calendar.month_abbr[i] for i in range(1, 13)]
mp.head()

In [ ]:
fig = px.bar(
    mp,
    x="hour",
    y="vehicles",
    animation_frame="Month",
    category_orders={"Month": MONTHS},
    range_y=[0, 7500],
    title="Traffic Pulse 2017 — weekday profile by month"
)
fig.show()

check_E(frame_budget_ms, smoothstep, n_frames, duration_s, compression, fig)

In [ ]:
# (given) The SAME data as small multiples — compare with the animation
fig2 = px.bar(mp, x="hour", y="vehicles", facet_col="Month", facet_col_wrap=6, category_orders={"Month": MONTHS}, height=420,
              title="Small multiples: all 12 months at once")
fig2.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1])); fig2.show()

In [ ]:
# Identify the month with the lowest mean traffic at 17:00 (for E3).
peak_17 = mp.loc[mp["hour"] == 17].set_index("Month")["vehicles"].reindex(MONTHS)
lowest_17_month = peak_17.idxmin()
print("Lowest 17:00 mean traffic:", lowest_17_month, f"({peak_17.min():,.0f} vehicles/hour)")
display(peak_17.sort_values())

**E3.** Run the month-identification cell immediately below the small-multiples chart to print the month with the lowest 17:00 mean. The small multiples are usually faster and more certain for this comparison because all months are visible simultaneously; animation requires the viewer to remember earlier frames, placing extra demand on working memory. This is an example of “eyes beat memory.”

**E4.** Without a fixed `range_y`, each month may be auto-scaled independently. Similar bar heights could then represent different traffic volumes, or a small change could look artificially large. This violates the principle of **congruence**: the visual scale should remain consistent across animated states.

**E5.** Smoothstep, `s(u) = 3u² − 2u³`, eases into and out of each transition. At `u = 0.25`, its value is `0.15625`, compared with `0.25` for linear motion, so the early movement is gentler. Its slope is zero at both endpoints, reducing abrupt starts and stops and making it easier to track the changing marks.

---
## Part F — Final recommendation (3 min) *(slides 15–16, 33–38)*
Write your recommendation to your manager.

**F1.** Give engineers a static hourly profile with weekday and holiday comparisons, plus a daily time-series overview and a zoomed detail view around candidate closure dates. Use filtering and small multiples or aligned event views to compare conditions. Avoid animation for the decision: static views make direct comparisons easier and preserve the evidence for review.

**F2.** For the lobby animation: (1) use a consistent frame rate and a clear, readable hold time for each hour; (2) keep the y-axis range fixed across every month so bar lengths remain comparable; and (3) label the current month/hour and provide playback controls or a pause option. Use smooth easing for transitions rather than abrupt jumps.

**F3.** This is **change blindness**: viewers may fail to notice a change when attention is elsewhere or when states change over time. Reduce it by using a blink-comparator or explicit before/after small multiples, and by highlighting the changed evening-peak mark with a clear annotation or visual cue.

---
## Score — run this cell

In [ ]:
summary()